# 🎙️ Indic ASR Benchmark for Punjabi Telephony Speech

This notebook benchmarks and compares the **Top 3 State-of-the-Art Indic ASR Models** side-by-side on Punjabi (`pa`) telephony audio conversations:

1. **Model 1: `bodhan-ai/indic-transcribe-core`**
   * *Architecture:* 1.2B parameter FastConformer + 24-layer Transformer Decoder (AI4Bharat + Bodhan AI + NVIDIA).
   * *Characteristics:* Trained on 1.3M+ hours of Indic speech; highest precision native Gurmukhi script output.

2. **Model 2: `ai4bharat/indic-conformer-600m-multilingual`**
   * *Architecture:* 600M parameter Conformer with Hybrid CTC / RNN-T decoder covering all 22 official languages.
   * *Characteristics:* Non-autoregressive, zero hallucination loops, fast inference.

3. **Model 3: `ai4bharat/indicconformer_stt_pa_hybrid_rnnt_large`**
   * *Architecture:* Language-specific Conformer-Large (~120M parameters) fine-tuned exclusively for Punjabi.
   * *Characteristics:* High sensitivity to Punjabi vocabulary, idioms, and phonetic traits.

---

### ⚡ Pipeline Overview
```
[Audio Directory (.wav/.mp3)]
             │
             ▼ (glob loop)
[Resample 16kHz Mono] ──► [Silero VAD (Remove Silence & Background Hum)]
                                      │
                      ┌───────────────┼───────────────┐
                      ▼               ▼               ▼
              [Bodhan Core 1.2B]  [Indic 600M]    [Indic PA Large]
                      │               │               │
                      └───────────────┼───────────────┘
                                      ▼
                       [Pandas DataFrame Comparison]
```

## 1. Environment Setup & Dependencies
Run the cell below to install the required libraries. For GPU environments (CUDA), ensure PyTorch is properly installed.

In [ ]:
# Install core packages
!pip install -q torch torchaudio transformers huggingface_hub soundfile librosa pandas tqdm tabulate

# Note for AI4Bharat NeMo Models (Required for Model 3 / IndicConformer NeMo):
# If NeMo is not already installed in your environment, install AI4Bharat's NeMo fork:
# !git clone https://github.com/AI4Bharat/NeMo.git && cd NeMo && git checkout nemo-v2 && pip install -e .
# Alternatively, install standard nemo-toolkit:
!pip install -q "nemo_toolkit[asr]" sentencepiece

## 2. Imports & Configuration
Specify the path to your audio directory containing the Punjabi telephone conversation files.

In [ ]:
import os
import sys
import glob
import tempfile
from pathlib import Path
import torch
import torchaudio
import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---------------- CONFIGURATION ----------------
# Set your directory path containing Punjabi telephone audio files
AUDIO_DIR = "/path/to/your/punjabi_telephone_audios"
AUDIO_EXTENSIONS = ["*.wav", "*.mp3", "*.flac", "*.m4a", "*.ogg"]

TARGET_SR = 16000     # All 3 models require 16kHz mono audio
LANGUAGE_CODE = "pa"  # Punjabi BCP-47 / ISO code

# VAD Parameters (Telephony audio tuning)
VAD_THRESHOLD = 0.5            # Speech detection threshold (0.0 to 1.0)
MIN_SPEECH_DURATION_MS = 250   # Discard voice bursts shorter than 250ms
MIN_SILENCE_DURATION_MS = 400  # Pause duration to consider segment separation

## 3. Voice Activity Detection (VAD) with Silero
Telephone audio often contains substantial non-speech parts: line noise, caller pauses, dial tones, and silence.
Silero VAD isolates active speech segments and removes dead air, preventing conformer sequence overflow and improving recognition accuracy.

In [ ]:
print("Loading Silero VAD model...")
vad_model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False
)
vad_model.to(device)
vad_model.eval()

(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

def preprocess_audio_with_vad(audio_path, target_sr=16000):
    """
    Loads an audio file, converts it to 16kHz mono, and extracts active speech.
    Returns:
        speech_tensor (torch.Tensor): Concatenated active speech tensor [1, T]
        total_dur (float): Total original duration in seconds
        speech_dur (float): Active speech duration in seconds
    """
    # Load with torchaudio
    wav, sr = torchaudio.load(str(audio_path))
    
    # Convert to mono if multi-channel
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)
        
    # Resample to 16kHz
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        wav = resampler(wav)
        sr = target_sr
        
    total_duration_sec = wav.shape[-1] / sr
    wav_mono = wav.squeeze(0)
    
    # Obtain speech timestamps
    speech_timestamps = get_speech_timestamps(
        wav_mono,
        vad_model,
        sampling_rate=sr,
        threshold=VAD_THRESHOLD,
        min_speech_duration_ms=MIN_SPEECH_DURATION_MS,
        min_silence_duration_ms=MIN_SILENCE_DURATION_MS
    )
    
    if len(speech_timestamps) == 0:
        # Fallback: if no speech detected (e.g. very quiet audio), use original
        speech_wav = wav
        speech_duration_sec = total_duration_sec
    else:
        # Concatenate all speech segments with a 150ms buffer
        chunks = [wav_mono[ts['start']:ts['end']] for ts in speech_timestamps]
        pad = torch.zeros(int(sr * 0.15))  # 150ms silence pad between chunks
        stitched = []
        for c in chunks:
            stitched.append(c)
            stitched.append(pad)
        speech_wav = torch.cat(stitched).unsqueeze(0)
        speech_duration_sec = sum(ts['end'] - ts['start'] for ts in speech_timestamps) / sr
        
    return speech_wav, total_duration_sec, speech_duration_sec

## 4. Load the 3 Models
We load all three models into memory once before looping through the dataset.

In [ ]:
# -------------------------------------------------------------------------
# MODEL 1: Bodhan AI / AI4Bharat Indic-Transcribe Core (1.2B)
# -------------------------------------------------------------------------
print("Loading Model 1: bodhan-ai/indic-transcribe-core...")
from huggingface_hub import snapshot_download

try:
    bodhan_dir = snapshot_download("bodhan-ai/indic-transcribe-core")
    if bodhan_dir not in sys.path:
        sys.path.insert(0, bodhan_dir)
    from indic_transcribe import IndicTranscribe
    model_bodhan = IndicTranscribe.from_pretrained(bodhan_dir)
    print("✅ Model 1 (Bodhan Indic-Transcribe Core) loaded successfully.")
except Exception as e:
    print(f"⚠️ Error loading Bodhan model: {e}")
    model_bodhan = None

# -------------------------------------------------------------------------
# MODEL 2: AI4Bharat IndicConformer Multilingual (600M)
# -------------------------------------------------------------------------
print("\nLoading Model 2: ai4bharat/indic-conformer-600m-multilingual...")
from transformers import AutoModel

try:
    model_indic_multi = AutoModel.from_pretrained(
        "ai4bharat/indic-conformer-600m-multilingual",
        trust_remote_code=True
    )
    model_indic_multi.eval()
    if torch.cuda.is_available():
        model_indic_multi = model_indic_multi.to(device)
    print("✅ Model 2 (IndicConformer 600M Multilingual) loaded successfully.")
except Exception as e:
    print(f"⚠️ Error loading IndicConformer 600M Multilingual: {e}")
    model_indic_multi = None

# -------------------------------------------------------------------------
# MODEL 3: AI4Bharat IndicConformer Punjabi-Specific Large (~120M)
# -------------------------------------------------------------------------
print("\nLoading Model 3: ai4bharat/indicconformer_stt_pa_hybrid_rnnt_large...")
try:
    import nemo.collections.asr as nemo_asr
    model_indic_pa = nemo_asr.models.ASRModel.from_pretrained(
        "ai4bharat/indicconformer_stt_pa_hybrid_rnnt_large"
    )
    model_indic_pa.freeze()
    if torch.cuda.is_available():
        model_indic_pa = model_indic_pa.to(device)
    # Set desired decoder: 'rnnt' (higher accuracy) or 'ctc' (faster)
    model_indic_pa.cur_decoder = "rnnt"
    print("✅ Model 3 (IndicConformer Punjabi-Specific Large) loaded successfully.")
except Exception as e:
    print(f"⚠️ Error loading IndicConformer Punjabi model via NeMo: {e}")
    print("Ensure 'nemo_toolkit[asr]' is installed.")
    model_indic_pa = None

## 5. Inference Wrapper Functions
Standardized inference wrappers to transcribe cleaned audio across all 3 models.

In [ ]:
def transcribe_bodhan(wav_tensor, sample_rate, temp_wav_path):
    """Transcribes using Bodhan Indic-Transcribe Core (1.2B)"""
    if model_bodhan is None:
        return "[Model 1 Not Loaded]"
    try:
        # Bodhan model accepts file path
        text = model_bodhan(str(temp_wav_path), lang=LANGUAGE_CODE)
        return text.strip() if isinstance(text, str) else str(text)
    except Exception as e:
        return f"[Error: {e}]"

def transcribe_indic_multi(wav_tensor, sample_rate):
    """Transcribes using AI4Bharat IndicConformer 600M Multilingual"""
    if model_indic_multi is None:
        return "[Model 2 Not Loaded]"
    try:
        # IndicConformer Multilingual takes tensor input [1, T]
        inp = wav_tensor.to(device)
        # Choose 'ctc' or 'rnnt'
        with torch.no_grad():
            text = model_indic_multi(inp, LANGUAGE_CODE, "ctc")
        return text.strip() if isinstance(text, str) else str(text)
    except Exception as e:
        return f"[Error: {e}]"

def transcribe_indic_pa(wav_tensor, sample_rate, temp_wav_path):
    """Transcribes using AI4Bharat IndicConformer Punjabi-Specific Large"""
    if model_indic_pa is None:
        return "[Model 3 Not Loaded]"
    try:
        with torch.no_grad():
            transcription = model_indic_pa.transcribe(
                [str(temp_wav_path)],
                batch_size=1,
                language_id=LANGUAGE_CODE
            )
            if isinstance(transcription, list) and len(transcription) > 0:
                return transcription[0].strip()
            return str(transcription).strip()
    except Exception as e:
        return f"[Error: {e}]"

## 6. Batch Inference Loop (Glob Directory)
Find all audio files in `AUDIO_DIR`, preprocess with VAD, and run all 3 models sequentially.

In [ ]:
# Discover all audio files matching extensions
audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(list(Path(AUDIO_DIR).glob(ext)))

audio_files = sorted(audio_files)
print(f"Found {len(audio_files)} audio file(s) in: {AUDIO_DIR}")

if len(audio_files) == 0:
    print("⚠️ No audio files found. Please verify your AUDIO_DIR path.")

results = []

# Create a temporary directory for VAD-processed audio files
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    
    for audio_path in tqdm(audio_files, desc="Transcribing Punjabi Audio"):
        filename = audio_path.name
        
        try:
            # 1. Voice Activity Detection (VAD) & Resampling
            clean_wav, total_sec, speech_sec = preprocess_audio_with_vad(audio_path, target_sr=TARGET_SR)
            
            # Save temp cleaned audio for file-based model interfaces
            temp_file = tmp_path / f"clean_{filename}.wav"
            torchaudio.save(str(temp_file), clean_wav, TARGET_SR)
            
            # 2. Run Model 1: Bodhan Indic-Transcribe Core (1.2B)
            out_bodhan = transcribe_bodhan(clean_wav, TARGET_SR, temp_file)
            
            # 3. Run Model 2: IndicConformer Multilingual (600M)
            out_indic_multi = transcribe_indic_multi(clean_wav, TARGET_SR)
            
            # 4. Run Model 3: IndicConformer Punjabi-Specific Large
            out_indic_pa = transcribe_indic_pa(clean_wav, TARGET_SR, temp_file)
            
            # Remove temp file
            if temp_file.exists():
                temp_file.unlink()
                
            results.append({
                "Filename": filename,
                "Total_Duration_s": round(total_sec, 2),
                "Speech_Duration_s": round(speech_sec, 2),
                "Speech_Ratio": f"{round((speech_sec / total_sec) * 100, 1)}%" if total_sec > 0 else "0%",
                "Bodhan_Core_1.2B": out_bodhan,
                "IndicConformer_Multi_600M": out_indic_multi,
                "IndicConformer_Punjabi_Large": out_indic_pa
            })
            
        except Exception as err:
            print(f"Error processing {filename}: {err}")
            results.append({
                "Filename": filename,
                "Total_Duration_s": 0,
                "Speech_Duration_s": 0,
                "Speech_Ratio": "0%",
                "Bodhan_Core_1.2B": f"[Failed: {err}]",
                "IndicConformer_Multi_600M": f"[Failed: {err}]",
                "IndicConformer_Punjabi_Large": f"[Failed: {err}]"
            })

## 7. Comparative Results in Pandas DataFrame
Display the side-by-side comparison of the transcriptions across all three models.

In [ ]:
df = pd.DataFrame(results)

# Add word counts for each model to inspect verbosity & missing words
if not df.empty:
    df["Words_Bodhan"] = df["Bodhan_Core_1.2B"].apply(lambda x: len(str(x).split()))
    df["Words_Multi"] = df["IndicConformer_Multi_600M"].apply(lambda x: len(str(x).split()))
    df["Words_PA_Specific"] = df["IndicConformer_Punjabi_Large"].apply(lambda x: len(str(x).split()))

# Configure Pandas display for clear Punjabi Gurmukhi text wrapping
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print(f"Total Transcriptions: {len(df)}")
df[["Filename", "Speech_Duration_s", "Bodhan_Core_1.2B", "IndicConformer_Multi_600M", "IndicConformer_Punjabi_Large"]].head(10)

## 8. Export Results
Save the output DataFrame to CSV and Excel formats for downstream evaluation or review.

In [ ]:
csv_output_path = "punjabi_asr_comparison_results.csv"
df.to_csv(csv_output_path, index=False, encoding="utf-8-sig")
print(f"Saved results to: {csv_output_path}")

# Optional: Export to Excel
try:
    excel_output_path = "punjabi_asr_comparison_results.xlsx"
    df.to_excel(excel_output_path, index=False)
    print(f"Saved Excel results to: {excel_output_path}")
except Exception as e:
    pass

## 9. (Optional) CER & WER Evaluation Against Ground Truth
If you have reference transcripts, you can calculate the Character Error Rate (CER) and Word Error Rate (WER) directly using this cell.

It applies **Unicode Normalization** (standardizing Gurmukhi tippi, bindi, addak, and vowel signs) so font and keyboard layout inconsistencies don't falsely inflate the error rate.

In [ ]:
!pip install -q editdistance jiwer unicodedata2
import unicodedata
import editdistance

def normalize_punjabi_text(text):
    """
    Normalizes Punjabi Unicode representation (NFC), strips non-lexical punctuation.
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    # Remove common punctuation
    for p in ["।", ".", ",", "?", "!", "-", "'", '"', "(", ")"]:
        text = text.replace(p, " ")
    # Collapse whitespace
    return " ".join(text.split()).strip()

def calculate_cer(ref, hyp):
    ref = normalize_punjabi_text(ref)
    hyp = normalize_punjabi_text(hyp)
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    return editdistance.eval(ref, hyp) / len(ref)

def calculate_wer(ref, hyp):
    ref_words = normalize_punjabi_text(ref).split()
    hyp_words = normalize_punjabi_text(hyp).split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return editdistance.eval(ref_words, hyp_words) / len(ref_words)

# Example usage if df contains a 'Ground_Truth' column:
if "Ground_Truth" in df.columns:
    for col in ["Bodhan_Core_1.2B", "IndicConformer_Multi_600M", "IndicConformer_Punjabi_Large"]:
        df[f"{col}_CER"] = df.apply(lambda row: calculate_cer(row["Ground_Truth"], row[col]), axis=1)
        df[f"{col}_WER"] = df.apply(lambda row: calculate_wer(row["Ground_Truth"], row[col]), axis=1)
        print(f"{col} -> Mean CER: {df[f'{col}_CER'].mean():.2%}, Mean WER: {df[f'{col}_WER'].mean():.2%}")